<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/13_evals_in_ci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13 · Evals in CI

An eval you run by hand is a demo. An eval that runs on every pull request is a test suite.

The difference is not technical — you have all the evaluators already — it is that nobody
remembers to run the notebook, and everybody notices a red check on their PR.

**New in this lesson:** the pytest integration, a GitHub Actions workflow, regression policy for
non-deterministic tests, and treating cost and latency as correctness constraints.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "openevals~=0.2.0" \
  "agentevals~=0.0.9" \
  "pytest"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-13-ci"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

---

## 1. Evals as pytest tests

`@pytest.mark.langsmith` turns an ordinary test into a LangSmith experiment: inputs, outputs,
and feedback are all logged, so a CI failure comes with a trace rather than just an assertion.

Write the file to disk exactly as it would live in your repo.

In [ ]:
!mkdir -p evals

In [ ]:
%%writefile evals/conftest.py
"""Shared fixtures. The agent under test is built once per session."""

import pytest

from deepagents import create_deep_agent
from langchain_core.tools import tool

MODEL = "langsmith:openai/gpt-5.6-luna"

ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

REFUND_POLICY = """
- Damaged on arrival: full refund or replacement, no time limit.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Refunds above $200 require human approval.
"""


@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by its 4-digit ID. Returns item, status, days since order, and price."""
    for o in ORDERS:
        if o["id"] == order_id:
            return (f"Order {o['id']}: {o['item']}, ${o['price']:.2f}, "
                    f"status={o['status']}, ordered {o['days_ago']} days ago")
    return f"No order {order_id}. Order IDs are 4 digits."


@tool
def get_refund_policy() -> str:
    """Return the refund policy. Consult before promising any remedy."""
    return REFUND_POLICY


@pytest.fixture(scope="session")
def agent():
    return create_deep_agent(
        model=MODEL,
        tools=[lookup_order, get_refund_policy],
        system_prompt=(
            "You are a customer support agent.\n"
            "Look up the order before answering, and note days since delivery.\n"
            "Read the refund policy and apply it literally.\n"
            "Faulty after 30 days means REPAIR ONLY - never offer a refund or replacement."
        ),
    )

In [ ]:
%%writefile evals/test_support_agent.py
"""Eval suite for the support agent.

Run:  pytest evals -v
Results appear in LangSmith as an experiment.
"""

import time

import pytest
from agentevals.trajectory.match import create_trajectory_match_evaluator
from langsmith import testing as t
from openevals import create_llm_as_judge

MODEL = "langsmith:openai/gpt-5.6-luna"

POLICY_RUBRIC = """
Grade this support reply for POLICY CORRECTNESS only. Ignore tone and length.

<policy>
- Damaged on arrival: full refund or replacement, no time limit.
- Faulty within 30 days: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Refunds above $200 require human approval.
</policy>

<facts>{inputs}</facts>
<reply>{outputs}</reply>

Score true only if the remedy offered is one the policy permits for these facts.
Score false if it offers a refund or replacement where only repair is allowed.
"""


def _answer(agent, question: str) -> str:
    return agent.invoke({"messages": [{"role": "user", "content": question}]})["messages"][-1].text


def _tools_called(agent, question: str) -> list[str]:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]


# --------------------------------------------------------------- unit-style
@pytest.mark.langsmith
@pytest.mark.parametrize(
    ("question", "expected_tool"),
    [
        ("What is the status of order 1042?", "lookup_order"),
        ("What is our policy on refunds after 30 days?", "get_refund_policy"),
    ],
)
def test_selects_the_right_tool(agent, question, expected_tool):
    """Single-step: the first decision is the right one. Fast, cheap, gates merges."""
    t.log_inputs({"question": question})
    called = _tools_called(agent, question)
    t.log_outputs({"tools_called": called})
    assert expected_tool in called


@pytest.mark.langsmith
def test_no_refund_after_thirty_days(agent):
    """The specific regression from lesson 10. Deterministic, so it gates merges."""
    question = "Order 1047 - the laptop stand wobbles. Can they get a refund?"
    t.log_inputs({"question": question})

    answer = _answer(agent, question)
    t.log_outputs({"answer": answer})

    lowered = answer.lower()
    offers_refund = "refund" in lowered and not any(
        phrase in lowered for phrase in ("no refund", "cannot refund", "not eligible for a refund",
                                         "repair only", "unable to offer a refund")
    )
    assert not offers_refund, f"Offered a refund on a 62-day-old fault:\n{answer}"


# --------------------------------------------------------------- integration-style
@pytest.mark.langsmith
def test_checks_policy_before_answering(agent):
    """Trajectory: the policy must be consulted, whatever else happens."""
    question = "Order 1043's desk lamp stopped working. Can we refund it?"
    t.log_inputs({"question": question})

    called = _tools_called(agent, question)
    t.log_outputs({"tools_called": called})

    assert "get_refund_policy" in called, f"Answered without reading the policy: {called}"
    assert "lookup_order" in called, f"Answered without looking up the order: {called}"


# --------------------------------------------------------------- judged (nightly)
@pytest.mark.langsmith
@pytest.mark.nightly
def test_policy_correctness_judged(agent):
    """LLM-judged, so non-deterministic. Runs nightly rather than gating merges."""
    facts = "Order 1047, laptop stand, faulty, delivered 62 days ago."
    t.log_inputs({"facts": facts})

    answer = _answer(agent, "Order 1047 - the laptop stand wobbles. Can they get a refund?")
    t.log_outputs({"answer": answer})

    judge = create_llm_as_judge(prompt=POLICY_RUBRIC, feedback_key="policy_correct", model=MODEL)
    verdict = judge(inputs=facts, outputs=answer)
    t.log_feedback(key="policy_correct", score=verdict["score"])

    assert verdict["score"], verdict["comment"]


# --------------------------------------------------------------- budgets
@pytest.mark.langsmith
def test_latency_budget(agent):
    """Correctness is not the only requirement anyone actually has."""
    start = time.time()
    _answer(agent, "What is the status of order 1042?")
    elapsed = time.time() - start

    t.log_outputs({"seconds": round(elapsed, 2)})
    assert elapsed < 45, f"A simple lookup took {elapsed:.1f}s"

In [ ]:
%%writefile evals/pytest.ini
[pytest]
markers =
    nightly: slow or non-deterministic; excluded from the merge gate

In [ ]:
# Run the merge-gating subset, exactly as CI would.
!cd evals && python -m pytest . -v -m "not nightly" 2>&1 | tail -25

Those results are now in LangSmith as an experiment — inputs, outputs, and feedback per test.
A failing CI run links to the trace that failed, so the first debugging step is reading what the
agent actually did rather than trying to reproduce it locally.

> 📸 **`13-pytest-experiment.png`** — The LangSmith experiment view for a pytest run, showing one row per test with pass/fail status and the logged inputs and outputs.
>
> *Caption:* pytest results as a LangSmith experiment — each row links to its trace.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/13-pytest-experiment.png`

### 🧠 Checkpoint

Every test above calls a live model. Which should gate a merge, and which should run nightly?

<details><summary>Show answer</summary>

**Gate merges:** `test_selects_the_right_tool`, `test_no_refund_after_thirty_days`,
`test_checks_policy_before_answering`, `test_latency_budget`. All are deterministic assertions —
a tool name is in a list, a substring is absent, a duration is under a bound. They fail only
when behaviour actually changed.

**Nightly:** `test_policy_correctness_judged`. It has a second model in the loop, so it can
disagree with itself between runs. A merge gate that fails ~3% of the time for no reason trains
everyone to hit re-run without reading, which destroys the value of every *other* test in the
suite.

The principle: **gate on tests whose failure is always a real signal.** Run everything else on a
schedule, where a flake costs you a glance at a dashboard instead of blocking a colleague.

There is a cost angle too. A judged suite over 200 examples is real money on every push.

</details>

---

## 2. The GitHub Actions workflow

Two jobs: a fast gate on every PR, and a deep run on a schedule.

In [ ]:
!mkdir -p .github/workflows

In [ ]:
%%writefile .github/workflows/evals.yml
name: agent evals

on:
  pull_request:
  schedule:
    - cron: "0 3 * * *"      # nightly deep run
  workflow_dispatch:

jobs:
  gate:
    name: merge gate (deterministic)
    runs-on: ubuntu-latest
    timeout-minutes: 15
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip
      - run: pip install -r requirements-evals.txt
      - name: run gating evals
        env:
          # One secret. The gateway serves the models, so there is no provider key.
          LANGSMITH_API_KEY: ${{ secrets.LANGSMITH_API_KEY }}
          LANGSMITH_TRACING: "true"
          LANGSMITH_PROJECT: "ci-${{ github.event.pull_request.number || github.run_id }}"
        run: pytest evals -v -m "not nightly"

  nightly:
    name: full suite (judged)
    if: github.event_name == 'schedule' || github.event_name == 'workflow_dispatch'
    runs-on: ubuntu-latest
    timeout-minutes: 45
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip
      - run: pip install -r requirements-evals.txt
      - name: run every eval
        env:
          LANGSMITH_API_KEY: ${{ secrets.LANGSMITH_API_KEY }}
          LANGSMITH_TRACING: "true"
          LANGSMITH_PROJECT: "nightly-${{ github.run_id }}"
        run: pytest evals -v

`LANGSMITH_PROJECT` is per-PR, so each pull request's traces are grouped and easy to compare
against the baseline.

---

## 3. Regression policy for non-deterministic tests

You cannot assert equality against a model. So assert against a **baseline**, with a tolerance.

In [ ]:
from langsmith import Client

client = Client()

BASELINE_SCORE = 0.85     # measured from the current main branch
TOLERANCE = 0.05          # noise you are willing to absorb


def check_regression(new_score: float) -> tuple[bool, str]:
    """Compare against a baseline rather than an absolute target."""
    delta = new_score - BASELINE_SCORE
    if delta < -TOLERANCE:
        return False, f"REGRESSION: {new_score:.2f} vs baseline {BASELINE_SCORE:.2f} ({delta:+.2f})"
    if delta > TOLERANCE:
        return True, f"IMPROVED: {new_score:.2f} ({delta:+.2f}) - consider raising the baseline"
    return True, f"stable: {new_score:.2f} ({delta:+.2f})"


for candidate in [0.87, 0.84, 0.78, 0.93]:
    ok, message = check_regression(candidate)
    print(f"{'PASS' if ok else 'FAIL'}  {message}")

Two habits that make this work:

- **Store the baseline in the repo**, next to the tests, and update it in the same PR that
  improves the score. Then the baseline is code-reviewed like anything else.
- **Raise it when you improve.** A baseline that only ever gets loosened is a ratchet pointing
  the wrong way.

---

## 4. Cost and latency are correctness constraints

An agent that is 3% more accurate and four times more expensive has not improved. Measure it in
the same suite.

In [ ]:
import time

from deepagents import create_deep_agent

def measure(agent, questions):
    """Accuracy is not the only axis. Track what the change costs you."""
    start = time.time()
    total_messages = 0
    for q in questions:
        result = agent.invoke({"messages": [{"role": "user", "content": q}]})
        total_messages += len(result["messages"])
    elapsed = time.time() - start
    return {
        "seconds": round(elapsed, 1),
        "avg_messages": round(total_messages / len(questions), 1),
    }


QUESTIONS = [
    "What is the status of order 1042?",
    "Order 1047 wobbles - refund?",
]

cheap = create_deep_agent(
    model=MODEL, tools=[lookup_order, get_refund_policy],
    system_prompt="You are a support agent. Look up the order and check the policy.",
)

thorough = create_deep_agent(
    model=MODEL, tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=(
        "You are a support agent. Look up the order, search past tickets for context, "
        "check the refund policy, and double-check your reasoning before answering."
    ),
)

print("cheap:   ", measure(cheap, QUESTIONS))
print("thorough:", measure(thorough, QUESTIONS))

> 📸 **`13-compare-commits.png`** — The LangSmith comparison view across several experiments from different commits, showing score, latency, and token columns side by side.
>
> *Caption:* Tracking accuracy, latency, and cost together across commits.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/13-compare-commits.png`

### 🧠 Checkpoint

A change makes the agent 3% more accurate and 4× more expensive per request.

Did the suite pass?

<details><summary>Show answer</summary>

Only if the suite was told to care, which by default it is not.

Most eval suites measure a single accuracy number, so this change ships as a clear win — and the
bill arrives a month later, attributed to nothing in particular because no test ever mentioned
cost.

The fix is to make budgets explicit alongside correctness:

```python
assert accuracy >= BASELINE_ACCURACY - TOLERANCE
assert p95_latency < 8.0
assert avg_tokens < 1.2 * BASELINE_TOKENS
```

Whether 3% for 4× is acceptable is a product decision, not an engineering one — a fraud check
might take that trade happily, a chat assistant never would. **The suite's job is to make the
trade visible before merge rather than after the invoice.**

</details>

### ✍️ Exercise

Add a token-budget guard to the suite:

1. Capture a baseline token count for a representative question (`run.total_tokens` from the
   LangSmith run, or approximate with `count_tokens_approximately` over the messages).
2. Write a `test_token_budget` that fails when usage exceeds 1.5× the baseline.
3. Mark it so it gates merges — a cost regression is deterministic enough to block on.
4. Deliberately break it: add "think step by step in extensive detail" to the system prompt and
   confirm the test catches it.

Step 4 is the point. A guard you have never seen fail is a guard you cannot trust.

<details><summary>Show a solution</summary>

```python
# evals/test_budgets.py
import pytest
from langchain_core.messages.utils import count_tokens_approximately
from langsmith import testing as t

BASELINE_TOKENS = 2500   # measured on main; update in the PR that changes it
MULTIPLIER = 1.5


def _token_usage(agent, question: str) -> int:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return count_tokens_approximately(result["messages"])


@pytest.mark.langsmith
def test_token_budget(agent):
    question = "Order 1047 - the laptop stand wobbles. Can they get a refund?"
    t.log_inputs({"question": question})

    used = _token_usage(agent, question)
    limit = BASELINE_TOKENS * MULTIPLIER

    t.log_outputs({"tokens": used, "limit": limit})
    t.log_feedback(key="token_ratio", score=used / BASELINE_TOKENS)

    assert used < limit, (
        f"Used {used} tokens vs budget {limit:.0f} "
        f"({used / BASELINE_TOKENS:.1f}x baseline). Cost regression."
    )
```

</details>

---

## 📌 Key takeaways

- Evals that only run by hand rot. CI is what makes them a test suite.
- `@pytest.mark.langsmith` turns each test into an experiment, so a CI failure links to a trace.
- Gate merges on deterministic tests only; run judged tests nightly.
- A flaky merge gate trains people to ignore **all** your tests, not just the flaky one.
- Compare against a baseline with a tolerance — never an absolute equality.
- Keep the baseline in the repo and raise it when you improve, or it becomes a one-way ratchet.
- Cost and latency are correctness constraints; a suite that ignores them will bless a 4× bill.

---

## ➡️ Next

**[14 · Online evals and closing the loop](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/14_online_evals.ipynb)**

Your suite now defends against regressions you already know about. The last question is how you
find the failures you have not imagined yet — which only production traffic can tell you.